# Clase 14 · Notebook 01
# La imagen y la convolución: filtros, stride y padding

**Introducción al Deep Learning — Módulo 3, Unidad 1: Redes convolucionales (Parte I)**

Antes de construir una CNN, vamos a **ver con nuestros ojos** qué es una imagen para un ordenador y qué hace
exactamente una convolución.

1. Una imagen es una matriz de números.
2. Aplicar filtros (kernels): el mapa de características.
3. El **stride** (desplazamiento) y su efecto en el tamaño.
4. El **padding** (relleno) para conservar el tamaño.

> 💡 Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Una imagen es una matriz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
np.random.seed(42)

img = load_digits().images[0]    # un "0" de 8x8
print("Forma:", img.shape, "| valores entre", int(img.min()), "y", int(img.max()))

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("Imagen (8x8)")
ax[1].imshow(img, cmap="gray")
for (i, j), v in np.ndenumerate(img):
    ax[1].text(j, i, int(v), ha="center", va="center", color="#FF647E", fontsize=8)
ax[1].set_title("Valores de los píxeles")
plt.tight_layout(); plt.show()

## 2. Aplicar filtros: el mapa de características

Una **convolución** desliza un filtro por la imagen y, en cada posición, **multiplica y suma**.
Definimos la operación a mano y probamos varios filtros clásicos.

In [ ]:
def convolucion(imagen, filtro, stride=1):
    fh, fw = filtro.shape
    oh = (imagen.shape[0] - fh) // stride + 1
    ow = (imagen.shape[1] - fw) // stride + 1
    salida = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            region = imagen[i*stride:i*stride+fh, j*stride:j*stride+fw]
            salida[i, j] = np.sum(region * filtro)
    return salida

from scipy.ndimage import zoom
grande = zoom(img, 6, order=1)   # ampliamos para que se vea mejor

filtros = {
    "Bordes verticales":   np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    "Bordes horizontales": np.array([[-1,-2,-1],[0,0,0],[1,2,1]]),
    "Desenfoque (blur)":   np.ones((3,3))/9,
    "Realce (sharpen)":    np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]),
}
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(grande, cmap="gray"); axes[0].set_title("Original"); axes[0].axis("off")
for ax, (nombre, k) in zip(axes[1:], filtros.items()):
    ax.imshow(np.abs(convolucion(grande, k)), cmap="gray"); ax.set_title(nombre); ax.axis("off")
plt.tight_layout(); plt.show()

Cada filtro resalta algo distinto. En una CNN, **la red no usa filtros fijos: los aprende** sola durante
el entrenamiento. El resultado de aplicar un filtro es el **mapa de características**.

## 3. El stride (desplazamiento)

El stride es cuánto avanza el filtro en cada paso. Un stride mayor produce un mapa más pequeño.

In [ ]:
for s in [1, 2, 3]:
    salida = convolucion(grande, filtros["Bordes verticales"], stride=s)
    print(f"stride {s}:  entrada {grande.shape}  ->  salida {salida.shape}")

## 4. El padding (relleno)

Al aplicar un filtro 3×3 sin relleno, la salida es **2 píxeles más pequeña** en cada dimensión. El **zero
padding** añade un marco de ceros para conservar el tamaño original.

In [ ]:
k = filtros["Bordes verticales"]

sin_padding = convolucion(grande, k)                    # 'valid'
con_padding = convolucion(np.pad(grande, 1), k)         # 'same' (marco de ceros)

print("Entrada:           ", grande.shape)
print("Sin padding (valid):", sin_padding.shape, " <- se encoge")
print("Con padding (same): ", con_padding.shape, " <- conserva el tamaño")

## Resumen

- Una **imagen** es una matriz de píxeles (números).
- La **convolución** desliza un filtro (multiplica y suma) y produce un **mapa de características**.
- En una CNN los **filtros se aprenden**, no se diseñan a mano.
- El **stride** controla el avance del filtro (y reduce el tamaño de salida).
- El **padding** (zero padding / `same`) conserva el tamaño; sin él (`valid`) la salida se encoge.

➡️ En el **Notebook 02** veremos el **pooling** y estas operaciones ya en capas de Keras (`Conv2D`, `MaxPooling2D`).